In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DecimalType

GOLD_SCHEMA = "workspace.sme_risk_gold"
EXPORT_DIR  = "/Volumes/workspace/sme_risk_gold/app_export"
spark.sql("CREATE VOLUME IF NOT EXISTS workspace.sme_risk_gold.app_export")

def as_double(df):
    """ROUND() gives DECIMAL columns; store them as plain doubles so the app can do arithmetic."""
    return df.select([F.col(f.name).cast("double").alias(f.name) if isinstance(f.dataType, DecimalType)
                    else F.col(f.name) for f in df.schema.fields])

In [0]:
SMALL = ["rpt_key_metrics", "rpt_monthly_deregistrations", "rpt_survival_curve",
         "rpt_segment_risk", "rpt_profile_risk"]

def save_parquet(df, file_name):
    pdf = as_double(df).toPandas()
    pdf.attrs = {}            # serverless adds query metrics here; they can't be saved as JSON
    pdf.to_parquet(f"{EXPORT_DIR}/{file_name}.parquet", index=False)
    print(f"{file_name}.parquet written ({len(pdf):,} rows)")

for name in SMALL:
    save_parquet(spark.table(f"{GOLD_SCHEMA}.{name}"), name)

# red-flag counts used by the Risk tiers page
save_parquet(spark.sql(f"""
    SELECT lending_tier, lending_tier_order, red_flag,
           borrower_segment = 'Trading company' AS is_trading,
           COUNT(*) AS companies
    FROM {GOLD_SCHEMA}.rpt_company_lookup
    GROUP BY ALL
"""), "lending_tier_counts")

In [0]:
(as_double(spark.table(f"{GOLD_SCHEMA}.rpt_company_lookup").drop("lending_tier_order", "status_code"))
    .repartitionByRange(5, "acn")
    .sortWithinPartitions("acn")
    .write.mode("overwrite")
    .option("compression", "zstd")
    .parquet(f"{EXPORT_DIR}/rpt_company_lookup"))

In [0]:
def show(path):
    for f in dbutils.fs.ls(path):
        if f.name.endswith(".parquet"):
            print(f"{f.size / 1e6:8.1f} MB  {f.name}")

show(EXPORT_DIR)
show(f"{EXPORT_DIR}/rpt_company_lookup")